# Explore the Gaia archive (sandbox notebook)

In [2]:
import astropy.units as u
from astropy.coordinates import SkyCoord

from astroquery.gaia import Gaia

In [3]:
# Gaia.MAIN_GAIA_TABLE = "gaiadr2.gaia_source"  # Select Data Release 2
Gaia.MAIN_GAIA_TABLE = "gaiadr3.gaia_source"  # Reselect Data Release 3, default

## Look for Andromeda galaxy

In [3]:
# You can use any of these names: 'M31', 'Andromeda', 'NGC 224', 'Messier 31'
andromeda = SkyCoord.from_name("M31")
print(f"Andromeda coordinates: {andromeda.to_string('hmsdms')}")

# Search around Andromeda
width = u.Quantity(1.0, u.deg)
height = u.Quantity(1.0, u.deg)

result = Gaia.query_object_async(coordinate=andromeda, width=width, height=height)
print(f"\nFound {len(result)} sources in the search area")
result.pprint(max_lines=12, max_width=130)


Andromeda coordinates: 00h42m44.3299992s +41d16m07.5s
INFO: Query finished. [astroquery.utils.tap.core]

Found 50 sources in the search area
         dist             solution_id             designation         ... ebpminrp_gspphot_upper libname_gspphot
                                                                      ...          mag                          
--------------------- ------------------- --------------------------- ... ---------------------- ---------------
0.0019576553118872925 1636148068921376768 Gaia DR3 381266999950756352 ...                     --                
 0.002135181883167782 1636148068921376768 Gaia DR3 381266999950756480 ...                     --                
0.0021955970327573932 1636148068921376768 Gaia DR3 381266999957532672 ...                     --                
                  ...                 ...                         ... ...                    ...             ...
 0.004693533478637199 1636148068921376768 Gaia DR3 38126775586378201

## Count objects in Gaia DR3







In [4]:
query = """
SELECT COUNT(*) as objects_count
FROM gaiadr3.gaia_source
"""

job = Gaia.launch_job_async(query)
results = job.get_results()
print(f"Number of objects in the DB: {results['objects_count'][0]}")


INFO: Query finished. [astroquery.utils.tap.core]
Number of objects in the DB: 1811709771


## Count galaxies in Gaia DR3

For a 3D visualization, a galaxy needs a **distance estimate**:

- **With redshift** (`redshift_ugc`): distance from cosmology.
- **Very nearby** (no redshift, but reliable parallax): `parallax_over_error > 5` and `parallax > 0.01` mas (~within 100 kpc).
- **Not placable**: no redshift and no reliable parallax — only sky direction (`ra`, `dec`) is known.

We count two populations:

1. **Strict galaxies** (`non_single_star = 3`): resolved galaxies in the NSS catalogue (~58k).
2. **Galaxy candidates** (`galaxy_candidates`): extragalactic table with UGC redshifts (~1.3M with redshift).


### Strict galaxies (`non_single_star = 3`)

Resolved galaxies in the NSS catalogue (~58k). Mostly local: distance from parallax, not redshift.

In [5]:
query = """
SELECT COUNT(*) AS n
FROM gaiadr3.gaia_source AS g
WHERE g.non_single_star = 3
"""

job = Gaia.launch_job_async(query)
results = job.get_results()
print(f"Strict galaxies (total): {results['n'][0]:,}")

INFO: Query finished. [astroquery.utils.tap.core]
Strict galaxies (total): 58,215


In [8]:
query = """
SELECT COUNT(*) AS n
FROM gaiadr3.gaia_source AS g
JOIN gaiadr3.galaxy_candidates AS gc USING (source_id)
WHERE g.non_single_star = 3
  AND gc.redshift_ugc IS NOT NULL
"""

job = Gaia.launch_job_async(query)
results = job.get_results()
print(f"Strict galaxies with redshift: {results['n'][0]:,}")

INFO: Query finished. [astroquery.utils.tap.core]
Strict galaxies with redshift: 0


In [9]:
query = """
SELECT COUNT(*) AS n
FROM gaiadr3.gaia_source AS g
WHERE g.non_single_star = 3
  AND g.parallax_over_error > 5
  AND g.parallax > 0.01
"""

job = Gaia.launch_job_async(query)
results = job.get_results()
print(
    f"Strict galaxies very nearby (parallax S/N > 5, d < ~100 kpc): {results['n'][0]:,}"
)

INFO: Query finished. [astroquery.utils.tap.core]
Strict galaxies very nearby (parallax S/N > 5, d < ~100 kpc): 58,209


In [ ]:
query = """
SELECT COUNT(*) AS n
FROM gaiadr3.gaia_source AS g
LEFT JOIN gaiadr3.galaxy_candidates AS gc USING (source_id)
WHERE g.non_single_star = 3
  AND gc.redshift_ugc IS NULL
  AND NOT (g.parallax_over_error > 5 AND g.parallax > 0)
"""

job = Gaia.launch_job_async(query)
results = job.get_results()
print(f"Strict galaxies not placable in 3D (exclude from viz): {results['n'][0]:,}")

### Galaxy candidates (`galaxy_candidates`)

Extragalactic table with UGC redshift estimates (~1.3M with redshift). Better starting point for a distant-universe 3D map.

Prefer querying `galaxy_candidates` alone when redshift is enough. Avoid `JOIN gaiadr3.gaia_source` unless parallax is required — it is much heavier and may trigger Gaia archive 500 errors.


In [3]:
query = """
SELECT COUNT(*) AS n
FROM gaiadr3.galaxy_candidates
"""

job = Gaia.launch_job_async(query)
results = job.get_results()
print(f"Galaxy candidates (total): {results['n'][0]:,}")

INFO: Query finished. [astroquery.utils.tap.core]
Galaxy candidates (total): 4,842,342


In [4]:
query = """
SELECT COUNT(*) AS n
FROM gaiadr3.galaxy_candidates
WHERE redshift_ugc IS NOT NULL
"""

job = Gaia.launch_job_async(query)
results = job.get_results()
print(f"Galaxy candidates with redshift: {results['n'][0]:,}")

INFO: Query finished. [astroquery.utils.tap.core]
Galaxy candidates with redshift: 1,367,153


**Skipped:** counting `galaxy_candidates` with parallax requires cross-lookup in `gaia_source` (~1.8 billion rows). On the Gaia archive this often runs for 10+ minutes and ends with HTTP 500.

Use the **strict galaxies** section above for nearby galaxies via parallax. For `galaxy_candidates`, use `redshift_ugc` instead.


In [11]:
# Fast proxy: candidates without a redshift estimate.
# A few may still be placable via parallax, but that is rare in this table.
query = """
SELECT COUNT(*) AS n
FROM gaiadr3.galaxy_candidates
WHERE redshift_ugc IS NULL
"""

job = Gaia.launch_job_async(query)
results = job.get_results()
print(
    f"Galaxy candidates without redshift (not placable via cosmology): {results['n'][0]:,}"
)


INFO: Query finished. [astroquery.utils.tap.core]
Galaxy candidates without redshift (not placable via cosmology): 3,475,189


## Count stars in Gaia DR3

In [6]:
# Let's start with a basic count of likely stars
query = """
SELECT COUNT(*) as mw_stars_count
FROM gaiadr3.gaia_source
WHERE non_single_star IS NULL OR non_single_star = 0  -- Just count likely single stars
"""

job = Gaia.launch_job_async(query)
results = job.get_results()
print(
    f"Number of likely single stars: {results['mw_stars_count'][0]:,}"
)

INFO: Query finished. [astroquery.utils.tap.core]
Number of likely single stars: 1,810,896,084


## Count stars in Milky Way

In [7]:
# Let's try counting stars with positive parallax
query2 = """
SELECT COUNT(*) as mw_stars_count
FROM gaiadr3.gaia_source
WHERE (non_single_star IS NULL OR non_single_star = 0)
AND parallax > 0  -- Just add positive parallax constraint
"""

job = Gaia.launch_job_async(query2)
results = job.get_results()
print(
    f"Number of likely single stars with positive parallax: {results['mw_stars_count'][0]:,}"
)


INFO: Query finished. [astroquery.utils.tap.core]
Number of likely single stars with positive parallax: 1,109,514,651


## Filter Milky Way disk with galactocentric coordinates

Download a pre-filtered sample from the Gaia archive, compute 3D galactocentric
coordinates locally, then keep sources inside a galactic disk
(`r_cyl_gc < 20 kpc`, `|z_gc| < 1 kpc`).

In [10]:
import numpy as np

from helpers import add_galactocentric_coords, filter_galactic_disk, galactic_disk_mask

query_sample = """
SELECT TOP 10000
    source_id, ra, dec, parallax, parallax_over_error
FROM gaiadr3.gaia_source
WHERE (non_single_star IS NULL OR non_single_star = 0)
  AND parallax_over_error > 5
  AND parallax > 0.01
"""

job = Gaia.launch_job_async(query_sample)
sample = job.get_results()

print(f"Downloaded sample: {len(sample):,} sources")

with_coords = add_galactocentric_coords(sample)
n_valid = np.isfinite(with_coords["r_cyl_gc"]).sum()
in_disk_mask = galactic_disk_mask(with_coords, r_max_kpc=20.0, z_max_kpc=1.0)
in_disk = filter_galactic_disk(with_coords, r_max_kpc=20.0, z_max_kpc=1.0)

print(f"Sources with valid 3D coordinates: {n_valid:,}")
print(f"Sources in galactic disk (R < 20 kpc, |Z| < 1 kpc): {in_disk_mask.sum():,}")
print(f"Fraction in disk: {in_disk_mask.sum() / len(with_coords):.1%}")

in_disk[:5].pprint(max_lines=12)

INFO: Query finished. [astroquery.utils.tap.core]
Downloaded sample: 10,000 sources
Sources with valid 3D coordinates: 10,000
Sources in galactic disk (R < 20 kpc, |Z| < 1 kpc): 9,697
Fraction in disk: 97.0%
    source_id              ra         ...      r_cyl_gc          r_sph_gc    
                          deg         ...                                    
------------------ ------------------ ... ----------------- -----------------
137340723276237440 45.252041556600155 ... 9.192453089294021 9.199164521173097
137342093370833408 45.189956475248366 ... 8.493314408585514 8.493530909938999
137346392632994944  44.88126694656205 ... 8.881006778412468 8.883872286463799
137346564431685120 44.909108932055474 ... 8.559151311987497 8.559619369517335
137349519369183744   44.9100658705783 ... 9.054385133650152 9.059269186662663
